In [4]:
import sys
!{sys.executable} -m pip install pandas --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 7.9 MB/s eta 0:00:008.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 9.7 MB/s eta 0:00:000m eta 0:00:010:00:01

[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: /home/jannes/.local/share/pipx/venvs/notebook/bin/python -m pip install --upgrade pip


In [ ]:
"""
kmer_features.py
----------------
Reads a CSV with columns:
    Name, eIF3d_control_logTE, eIF3d_depletion_logTE,
    eIF4e_control_logTE, eIF4e_depletion_logTE,
    tx_length, utr3_length, cds_length, utr5_length, tx_sequence

For each transcript, extracts the three sub-sequences:
    5' UTR  : tx_sequence[0 : utr5_length]
    CDS     : tx_sequence[utr5_length : utr5_length + cds_length]
    3' UTR  : tx_sequence[utr5_length + cds_length :]
    Full    : tx_sequence

Then computes k-mer frequencies (k = 2..6) for every region,
normalising each k-mer count by the total number of k-mers in that region
so that frequencies sum to 1.

Output:
    kmer_features.csv  – one row per transcript, columns = Name +
                         all k-mer frequency features
"""

import itertools
import pandas as pd
from collections import Counter

#load data and drop NaNs─────────────────────────────────────────
CSV_PATH = "TE_eIF_depletion.csv"
df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.dropna(subset=["utr5_length", "cds_length", "utr3_length", "tx_sequence"])
df_raw = df_raw.reset_index(drop=True)

#extract sequence columns ─────────────────────────────────────────
def extract_regions(row):
    seq   = str(row["tx_sequence"])
    u5    = int(row["utr5_length"])
    cds   = int(row["cds_length"])

    utr5_seq = seq[:u5]
    cds_seq  = seq[u5 : u5 + cds]
    utr3_seq = seq[u5 + cds:]

    return {"utr5": utr5_seq, "cds": cds_seq, "utr3": utr3_seq, "full": seq}

#compute k-mer frequencies ─────────────────────────────────────
NUCLEOTIDES = "ACGT"

def all_kmers(k):
    return ["".join(p) for p in itertools.product(NUCLEOTIDES, repeat=k)]
    #returns sorted list of all possible k-mers over {A,C,G,T}

def kmer_freq(sequence, k):
    counts = Counter(sequence[i:i+k] for i in range(len(sequence) - k + 1))
    total  = sum(counts.values())
    return {kmer: counts.get(kmer, 0) / total if total > 0 else 0.0 for kmer in all_kmers(k)}
    #returns a dict {kmer: frequency}.  Missing k-mers get 0.

#build dataframe ─────────────────────────────────────────────────────
regions      = ["utr5", "cds", "utr3"]
k_range      = range(2, 7)

feature_rows = []

for _, row in df_raw.iterrows():
    seqs     = extract_regions(row)
    feat_row = {"Name": row["Name"]}

    for region in regions:
        for k in k_range:
            freqs = kmer_freq(seqs[region], k)
            for kmer, freq in freqs.items():
                col_name = f"{region}_k{k}_{kmer}"
                feat_row[col_name] = freq

    feature_rows.append(feat_row)

df_kmers = pd.DataFrame(feature_rows)

#save ───────────────────────────────────────────────────────────────────
OUT_PATH = "kmer_features.csv"
df_kmers.to_csv(OUT_PATH, index=False)

print(f"  Transcripts : {len(df_kmers)}")
print(f"  Total cols  : {len(df_kmers.columns)}  "
      f"(4 TE columns + 1 Name + {len(df_kmers.columns)-5} k-mer features)")
print(f"  Saved to    : {OUT_PATH}")

In [1]:
# ── Quick sanity check ────────────────────────────────────────────────────────
for region in regions:
    for k in k_range:
        cols = [c for c in df_kmers.columns if c.startswith(f"{region}_k{k}_")]
        row_sums = df_kmers[cols].iloc[0].sum()
        status = "✓" if abs(row_sums - 1.0) < 1e-6 else "✗"
        print(f"  {status}  {region} k={k} freq sum (first transcript): {row_sums:.4f}")

NameError: name 'regions' is not defined

In [17]:
"""
Streaming one-hot encoding for Kozak positions:
-3, -2, -1, +4, +5

Drops T as reference category.
Outputs 3x5=15 predictors per transcript.
"""

import pandas as pd
import csv

CSV_PATH = "TE_eIF_depletion.csv"
OUT_PATH = "kozak_onehot.csv"

positions = {
    "-3": -3,
    "-2": -2,
    "-1": -1,
    "+4":  3,
    "+5":  4,
}

NUCLEOTIDES = ["A", "C", "G"]  # T is reference

# ─────────────────────────────────────────
# load
# ─────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["utr5_length", "tx_sequence"])
df = df.reset_index(drop=True)

print(f"Loaded {len(df)} transcripts.", flush=True)

# ─────────────────────────────────────────
# streaming write
# ─────────────────────────────────────────
with open(OUT_PATH, "w", newline="") as f:
    writer = None

    for i, (_, row) in enumerate(df.iterrows()):
        seq = str(row["tx_sequence"])
        u5  = int(row["utr5_length"])

        feat_row = {"Name": row["Name"]}

        valid = True

        for label, offset in positions.items():
            idx = u5 + offset

            if idx < 0 or idx >= len(seq):
                valid = False
                base = None
            else:
                base = seq[idx]

            for nuc in NUCLEOTIDES:
                colname = f"{label}_{nuc}"
                feat_row[colname] = 1 if base == nuc else 0

        # Initialize header once
        if writer is None:
            writer = csv.DictWriter(f, fieldnames=feat_row.keys())
            writer.writeheader()

        writer.writerow(feat_row)

        if (i + 1) % 1000 == 0:
            print(f"[{i+1}] processed", flush=True)

print("Done.")
print(f"Saved to {OUT_PATH}")

Loaded 11112 transcripts.
[1000] processed
[2000] processed
[3000] processed
[4000] processed
[5000] processed
[6000] processed
[7000] processed
[8000] processed
[9000] processed
[10000] processed
[11000] processed
Done.
Saved to kozak_onehot.csv


In [25]:
import pandas as pd
import csv

# ===============================
# CONFIG
# ===============================
INPUT_CSV = "TE_eIF_depletion.csv"
OUTPUT_CSV = "dicodon_density.csv"

# 17 validated inhibitory dicodons (DNA notation)
INHIBITORY_PAIRS = {
    "AGGCGA","AGGCGG","ATACGA","ATACGG",
    "CGAATA","CGACCG","CGACGA","CGACGG","CGACTG","CGAGCG",
    "CTCATA","CTCCCG",
    "CTGATA","CTGCCG","CTGCGA",
    "CTTCTG",
    "GTACCG"
}

VALID_BASES = set("ATCG")

# ===============================
# HELPERS
# ===============================
def log(logger, name, issue, detail=""):
    msg = f"[{issue}] {name}: {detail}"
    print(msg)
    logger.write(msg + "\n")

# ===============================
# PROCESS
# ===============================
df = pd.read_csv(INPUT_CSV)
df = df.dropna().reset_index(drop=True)
total     = len(df)

with open(OUTPUT_CSV, "w", newline="") as out:
    writer = csv.writer(out)
    writer.writerow(["Name", "dicodon_count", "dicodon_density"])

    for idx, row in df.iterrows():
        name = row["Name"]

        seq = str(row["tx_sequence"]).upper().replace("U", "T")
        cds = seq[u5 : u5 + cds_len]

        remainder = len(cds) % 3
        if remainder != 0:
            cds = cds[:-remainder]

        # -----------------------------------------------
        # 9. Count inhibitory dicodons, skip any with N
        # -----------------------------------------------
        codons = [cds[i:i+3] for i in range(0, len(cds), 3)]
        cds_codons = len(codons)

        n_skipped_dicodons = 0
        count = 0
        for i in range(cds_codons - 1):
            dicodon = codons[i] + codons[i+1]
            if "N" in dicodon:
                n_skipped_dicodons += 1
                continue
            if dicodon in INHIBITORY_PAIRS:
                count += 1

        possible_positions = (cds_codons - 1) - n_skipped_dicodons
        density = count / possible_positions if possible_positions > 0 else 0.0

        writer.writerow([name, count, round(density, 6)])

print(f"Results written to {OUTPUT_CSV}")

Results written to dicodon_density.csv
